# PAIEC: embed sampled items with Qwen3-Embedding-4B
Settings: Accelerator = GPU (T4 x2 preferred), Internet = On, Add-ons > Secrets: `HF_TOKEN` attached to this notebook (token must be able to read `aims-foundations/measurement-db`).
Set `SMOKE = True` for the quick check (3 small benchmarks, 50 items each), `False` for the full run. Re-running resumes.

In [ ]:
SMOKE = False
PER_BENCH = 2000

In [ ]:
!pip install -q -U "transformers>=4.51" accelerate huggingface_hub pandas pyarrow
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
import glob, os, subprocess, sys, torch
n_gpu = max(1, torch.cuda.device_count())
script = glob.glob('/kaggle/input/**/kaggle_embed_qwen3.py', recursive=True)[0]
base = dict(os.environ, QWEN_MODEL='Qwen/Qwen3-Embedding-4B', PER_BENCH=str(PER_BENCH), MAX_LEN='256', SMOKE='1' if SMOKE else '0')
procs = [subprocess.Popen([sys.executable, script], env=dict(base, CUDA_VISIBLE_DEVICES=str(i), SHARD=f'{i}/{n_gpu}'),
                          stdout=open(f'/kaggle/working/shard{i}.log', 'w'), stderr=subprocess.STDOUT) for i in range(n_gpu)]
print('GPUs', n_gpu, 'pids', [p.pid for p in procs])

In [ ]:
import time
out = '/kaggle/working/emb_qwen_smoke' if SMOKE else '/kaggle/working/emb_qwen'
while any(p.poll() is None for p in procs):
    time.sleep(60); print(len(glob.glob(out + '/*.npz')), 'files done; exit codes', [p.poll() for p in procs], flush=True)
for i in range(n_gpu): print(f'--- shard {i}\n' + open(f'/kaggle/working/shard{i}.log').read()[-1500:])

In [ ]:
import numpy as np
fs = sorted(glob.glob(out + '/*.npz')); tot = 0
for f in fs[:5]:
    z = np.load(f); E = z['emb'].astype(np.float32); tot += len(E)
    print(os.path.basename(f), E.shape, 'finite', bool(np.isfinite(E).all()), 'norm range', float(np.linalg.norm(E, axis=1).min()), float(np.linalg.norm(E, axis=1).max()))
print(len(fs), 'benchmarks written')